In [ ]:
# title: MapBiomas Soil
# subtitle: 24. Temporal Replication of SOC Samples
# author: Alessandro Samuel-Rosa and Taciara Zborowski Horst
# data: 2025

In [52]:
# Define the name of the FeatureCollection
fc_name = 'c03_soc_v2025_11_26'

In [53]:
# Load required libraries
import ee

# Initialize the Earth Engine API
# ee.Authenticate()
ee.Initialize(project='mapbiomas-solos-workspace')

# Define the path to the FeatureCollection directory
fc_dir = 'projects/mapbiomas-workspace/SOLOS/AMOSTRAS/MATRIZES/collection3/'
fc_path = f'{fc_dir}{fc_name}'
print(f'FeatureCollection path: {fc_path}')

# Load the FeatureCollection from the specified path
fc = ee.FeatureCollection(fc_path)

# Print the number of features in the FeatureCollection
num_features = fc.size().getInfo()
print(f'Number of features in the FeatureCollection: {num_features}')
# 30615 features out of 31047 in the original CSV. This means that 432 features were lost.

# Print the columns of the FeatureCollection
print(fc.first().propertyNames().getInfo())

# Print the minimum and maximum values of the 'vegNatural' property
print(fc.aggregate_min('vegNatural').getInfo())
# Min: 0 (OK)
print(fc.aggregate_max('vegNatural').getInfo())
# Max: 58 (OK)

FeatureCollection path: projects/mapbiomas-workspace/SOLOS/AMOSTRAS/MATRIZES/collection3/c03_soc_v2025_11_26
Number of features in the FeatureCollection: 30615
['latossolo_metamorficas', 'Amazonas_Solimoes_Provincia', 'year', 'agropecuaria', 'Amazonia_Provincia', 'koppen_l2_Cf', 'koppen_l2_Cs', 'raso_plutonica', 'mb_ndvi_median_decay', 'Wetsols', 'cti', 'ARGISSOLO', 'ORGANOSSOLO', 'Savana_Estepica', 'pantanal_neossolo_quartzarenico', 'id', 'argissolo_sedimentos', 'koppen_l2_Cw', 'Plinthosols', 'Floresta_Ombrofila_Densa', 'Ferralsols', 'Argisols', 'GLEISSOLO', 'Borborema_Provincia', 'mb_savi_median_decay', 'latossolo_plutonicas', 'Floresta_Estacional_Decidual', 'mb_fireRecurrence', 'carbono_gm2', 'slope', 'Floresta_Ombrofila_Aberta', 'argila_000_030cm', 'lavourasPerene', 'Floresta_Estacional_Sempre_Verde', 'Formacao_Pioneira', 'PLANOSSOLO', 'eastness', 'formacaoSavanica', 'afloramento', 'Parana_Provincia', 'formacaoCampestre', 'Caatinga', 'Area_Estavel', 'Reconcavo_Tucano_Jatoba_Provinc

In [54]:
# Create a list with natural vegetation classes:
properties = [
    'formacaoFlorestal', 'outrasFormacoesFlorestais', 'formacaoCampestre', 'formacaoSavanica',
    'campoAlagadoAreaPantanosa', 'restingas', 'afloramento']

# Create a column that contains the sum of the values (ages) of the properties
fc = fc.map(lambda feature: feature.set('sum', ee.Number(0)))
for property in properties:
    fc = fc.map(lambda feature: feature.set('sum', ee.Number(feature.get('sum')).add(feature.get(property))))

# Print the minimum and maximum values (ages) of the 'sum' property
print(fc.aggregate_min('sum').getInfo()) # zero means antropized area
print(fc.aggregate_max('sum').getInfo()) # max should be up to 60 (OK)

0
58


In [55]:
# Print the minimum and maximum values of the 'mb_edge' property
print(fc.aggregate_min('mb_edges').getInfo()) # min should be 0
print(fc.aggregate_max('mb_edges').getInfo()) # max should be 7000 m+

0
7707


In [56]:
# Identify the features that meet the following criteria: 
# - vegNatural > 0
# - year > 2004
# - sum of the values (ages) of the properties is greater than 39
# - mb_edges > 200
# Then print the result.  This is to guarantee that we are selecting stable natural features.
fc_stable = fc.filter(ee.Filter.gt('sum', 39)).filter(ee.Filter.gt('year', 2004)).filter(ee.Filter.gt('vegNatural', 0)).filter(ee.Filter.gt('mb_edges', 200))
print(f"Stable, undisturbed natural features found: {fc_stable.size().getInfo()}")
# 2310 features found.

Stable, undisturbed natural features found: 2310


In [57]:
# Print the minimum and maximum values of the 'year' property
print(fc_stable.aggregate_min('year').getInfo()) # Should be 2005
print(fc_stable.aggregate_max('year').getInfo()) # Should be up to 2024

2005
2023


In [58]:
# Print the minimum and maximum values of the 'vegNatural' property
print(fc_stable.aggregate_min('vegNatural').getInfo()) # Should be 40
print(fc_stable.aggregate_max('vegNatural').getInfo()) # Should be up to 60

40
58


In [59]:
# Print the minimum and maximum values of the 'agropecuaria' property
print(fc_stable.aggregate_min('agropecuaria').getInfo()) # Should be 0
print(fc_stable.aggregate_max('agropecuaria').getInfo()) # Should be 0

0
0


In [60]:
# Create replicas of the stable features by adjusting the 'year' property

# Subtract 10 from the 'year' property
fc_stable_10 = fc_stable.map(
  lambda feature: feature
    .set('year', ee.Number(feature.get('year')).subtract(10))
    .set('id', ee.String('trep10-').cat(feature.get('id')))
  )

# Subtract 20 from the 'year' property
fc_stable_20 = fc_stable.map(
  lambda feature: feature
    .set('year', ee.Number(feature.get('year')).subtract(20))
    .set('id', ee.String('trep20-').cat(feature.get('id')))
  )
							 
# Stack the three FeatureCollections: fc_filter, fc_stable_10, fc_stable_20
fc_stacked = fc.merge(fc_stable_10).merge(fc_stable_20)

# Print the number of features in the stacked FeatureCollection
print(f"Total samples after replication: {fc_stacked.size().getInfo()}")
# 35235 features

# Verification: Print the first few IDs to check the prefix
print("Sample IDs from -10 replica:")
print(fc_stable_10.limit(5).aggregate_array('id').getInfo())

Total samples after replication: 35235
Sample IDs from -10 replica:
['trep10-ctb0019-B', 'trep10-ctb0019-B', 'trep10-ctb0019-B', 'trep10-ctb0019-F10', 'trep10-ctb0019-F10']


In [61]:
# Row order the stacked FeatureCollection by the 'id' property
fc_stacked = fc_stacked.sort('id')

# Print the 'id' and 'year' properties of the first 10 features in the stacked
# FeatureCollection
print(fc_stacked.limit(10).aggregate_array('id').getInfo())
print(fc_stacked.limit(10).aggregate_array('year').getInfo())

['ctb0003-sm-dnos-001', 'ctb0003-sm-dnos-002', 'ctb0003-sm-dnos-003', 'ctb0003-sm-dnos-004', 'ctb0003-sm-dnos-005', 'ctb0003-sm-dnos-006', 'ctb0003-sm-dnos-007', 'ctb0003-sm-dnos-008', 'ctb0003-sm-dnos-009', 'ctb0003-sm-dnos-010']
[2009, 2009, 2009, 2009, 2009, 2009, 2009, 2009, 2009, 2009]


In [62]:
# Print the minimum and maximum values of the 'year' property
print(fc_stacked.aggregate_min('year').getInfo()) # Should be 2005 - 20 = 1985
print(fc_stacked.aggregate_max('year').getInfo()) # Should be up to 2024

1985
2024


In [63]:
# Import geemap to create an interactive map
import geemap

# Create a map object
map = geemap.Map()

# Add satellite basemap
# map.add_basemap('SATELLITE')
map.add_basemap('HYBRID')

# Add the FeatureCollection layers as before
map.addLayer(fc, {'color': 'blue'}, 'Original')
map.addLayer(fc_stable_10, {'color': 'red'}, 'Stable')

# Display the map
map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [64]:
# Save the FeatureCollection to a new asset
import datetime

# Asset folder
fc_dir = 'projects/mapbiomas-workspace/SOLOS/AMOSTRAS/ORIGINAIS/collection3/'
# Asset name
today = datetime.datetime.now().strftime("%Y_%m_%d")
fc_name = today + '_soildata_soc_trep'
fc_path = f'{fc_dir}{fc_name}'
print(f'FeatureCollection path: {fc_path}')

# Keep only the following columns:
soc_cols = [
  'id', 'year', 'profundidade', 'carbono_gm2', 'carbono_gm2_qmap',
  'IFN_index', 'YEAR_index', 'PSEUDOROCK_index', 'PSEUDOSAND_index']
fc_stacked = fc_stacked.select(soc_cols)
# Rename column: year --> ano
fc_stacked = fc_stacked.map(lambda feature: feature.set('ano', feature.get('year')))
# Update soc_cols to include 'ano' instead of 'year'
soc_cols_final = [
  'id', 'ano', 'profundidade', 'carbono_gm2', 'carbono_gm2_qmap',
  'IFN_index', 'YEAR_index', 'PSEUDOROCK_index', 'PSEUDOSAND_index']
fc_stacked = fc_stacked.select(soc_cols_final)
# Print the columns of the final FeatureCollection
print(fc_stacked.first().propertyNames().getInfo())

# Export the FeatureCollection
task = ee.batch.Export.table.toAsset(
  collection=fc_stacked, description='ASR: Soil Samples with Time Replicas', assetId=fc_path)
task.start()
print(task.status())

FeatureCollection path: projects/mapbiomas-workspace/SOLOS/AMOSTRAS/ORIGINAIS/collection3/2025_11_26_soildata_soc_trep
['system:index', 'ano', 'PSEUDOROCK_index', 'IFN_index', 'carbono_gm2_qmap', 'PSEUDOSAND_index', 'id', 'profundidade', 'carbono_gm2', 'YEAR_index']
{'state': 'READY', 'description': 'ASR: Soil Samples with Time Replicas', 'priority': 100, 'creation_timestamp_ms': 1764199655512, 'update_timestamp_ms': 1764199655512, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'PKFRRX42EEYOHR7G4EU7XARY', 'name': 'projects/mapbiomas-solos-workspace/operations/PKFRRX42EEYOHR7G4EU7XARY'}
